In [1]:
import os
import json
import logging
import torch
import librosa
import gigaam
import soundfile as sf
import numpy as np
from pathlib import Path
from collections import defaultdict
from pyannote.audio import Pipeline, Inference, Model
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("audio_pipeline")


W0812 02:18:22.077000 36644 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\Great_Ded\Desktop\univer\ML\yandex_project\pipeline\pipenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CONFIG_PATH = Path("config.txt")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл с токеном: {CONFIG_PATH.resolve()}. "
        "Создайте config.txt с HF-токеном внутри."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    HF_TOKEN = f.read().strip()

if not HF_TOKEN:
    logger.warning("config.txt пустой — запросы к приватным моделям HuggingFace могут не пройти")


In [3]:
class SileroVAD:
    """Silero VAD с оптимизированными параметрами из Optuna"""

    def __init__(
        self,
        threshold: float = 0.3346,
        min_speech_duration_ms: int = 294,
        min_silence_duration_ms: int = 217,
        speech_pad_ms: int = 160,
        neg_threshold: float = 0.3159,
        sampling_rate: int = 16000,
    ):
        self.threshold = threshold
        self.min_speech_duration_ms = min_speech_duration_ms
        self.min_silence_duration_ms = min_silence_duration_ms
        self.speech_pad_ms = speech_pad_ms
        self.neg_threshold = neg_threshold
        self.sampling_rate = sampling_rate

        self.model = load_silero_vad()

    def process(self, wav_file: np.ndarray) -> list[tuple[float, float]]:
        """Возвращает список (start, end) в секундах"""
        speech_timestamps = get_speech_timestamps(
            wav_file,
            self.model,
            sampling_rate=self.sampling_rate,
            threshold=self.threshold,
            min_speech_duration_ms=self.min_speech_duration_ms,
            min_silence_duration_ms=self.min_silence_duration_ms,
            speech_pad_ms=self.speech_pad_ms,
            return_seconds=True,
            neg_threshold=self.neg_threshold,
        )

        return [(seg["start"], seg["end"]) for seg in speech_timestamps]


In [4]:
class PyAnnoteDiarizer:
    """Диаризация на основе pyannote/speaker-diarization-community-1"""

    def __init__(self, device: torch.device, hf_token: str = ""):
        self.hf_token = hf_token or os.environ.get("HF_TOKEN", "")
        self.device = device if isinstance(device, torch.device) else torch.device(device)

        self.pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=self.hf_token or None,
        )
        self.pipeline.to(self.device)

    def process(self, wav_file: np.ndarray, sr: int) -> dict[str, list[tuple[float, float]]]:
        """Возвращает словарь {speaker_id: [(start, end), ...]}"""

        waveform = torch.from_numpy(wav_file).unsqueeze(0)

        with torch.inference_mode():
            result = self.pipeline({"waveform": waveform, "sample_rate": sr})
        ann = getattr(result, "speaker_diarization", result)

        speaker_segments = defaultdict(list)
        for segment, _, speaker in ann.itertracks(yield_label=True):
            speaker_segments[speaker].append((segment.start, segment.end))

        return dict(speaker_segments)


In [5]:
class GigaAmASR:
    """ASR модель на основе модели GigaAM"""

    def __init__(self, model_name: str, device: torch.device):
        self.model_name = model_name
        self.device = device
        self.model = gigaam.load_model(
            self.model_name,
            device=str(device),
        )

    def transcribe_audio_gigaam(self, audio_path: str) -> dict:
        """Проводит транскрибацию аудиодорожки"""
        with torch.inference_mode():
            result = self.model.transcribe(audio_path, word_timestamps=True)

        text = result.text

        word_timings = [
            {"word": word.text, "start": word.start, "end": word.end}
            for word in result.words
        ]

        return {"text": text, "words": word_timings}


In [ ]:
class AudioProcessingPipeline:
    MIN_EMBEDDING_SEGMENT_SEC = 0.5
    MIN_ASR_SEGMENT_SEC = 0.02
    MERGE_ADJACENT_GAP_SEC = 0.01
    GPU_CLEAR_EVERY_N_SEGMENTS = 5

    def __init__(self, hf_token: str, model_name: str, sample_rate: int = 16000):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        self.sample_rate = sample_rate

        self.vad = SileroVAD(sampling_rate=self.sample_rate)
        self.diarizer = PyAnnoteDiarizer(device=self.device, hf_token=hf_token)
        self.asr = GigaAmASR(model_name=self.model_name, device=self.device)

        self.temp_dir = Path("temp_audio_segments")
        self.temp_dir.mkdir(parents=True, exist_ok=True)

        self.embedding_model = Model.from_pretrained(
            "pyannote/embedding",
            token=hf_token,
        )
        self.embedding_inference = Inference(
            self.embedding_model,
            device=self.device,
            window="whole",
        )


    def _create_speaker_embedding(
        self,
        audio: np.ndarray,
        speaker_segments: dict[str, list[tuple[float, float]]],
        sr: int,
    ) -> dict[str, list[float] | None]:
        """Создаёт эмбеддинги для каждого спикера на основе всех его сегментов"""
        speaker_emb: dict[str, list[float] | None] = {}

        for speaker, segments in speaker_segments.items():
            collection = []

            for begin, end in segments:
                if end - begin < self.MIN_EMBEDDING_SEGMENT_SEC:
                    continue

                start_sample = int(begin * sr)
                end_sample = int(end * sr)
                cut_audio = audio[start_sample:end_sample]

                waveform = torch.from_numpy(cut_audio).unsqueeze(0)

                with torch.inference_mode():
                    embedding = self.embedding_inference(
                        {"waveform": waveform, "sample_rate": sr}
                    )

                collection.append(embedding.tolist())

            if collection:
                collection_np = np.array(collection)
                mean_emb = np.mean(collection_np, axis=0)
                norm = np.linalg.norm(mean_emb)
                if norm > 0:
                    mean_emb = mean_emb / norm
                speaker_emb[speaker] = mean_emb.tolist()
            else:
                logger.warning(
                    "Спикер %s: нет сегментов длиннее %.2fс — эмбеддинг не построен",
                    speaker, self.MIN_EMBEDDING_SEGMENT_SEC,
                )
                speaker_emb[speaker] = None

        return speaker_emb


    def _merge_segments_quality(
        self,
        vad_segments: list[tuple[float, float]],
        speaker_segments: dict[str, list[tuple[float, float]]],
    ) -> list[dict]:
        """Объединяет тайминги VAD и диаризации в общий список сегментов со спикерами"""
        merged = []
        all_boundaries = set()

        for start, end in vad_segments:
            all_boundaries.add(start)
            all_boundaries.add(end)

        for segments in speaker_segments.values():
            for start, end in segments:
                all_boundaries.add(start)
                all_boundaries.add(end)

        all_boundaries = sorted(all_boundaries)

        intervals = [
            (all_boundaries[i], all_boundaries[i + 1])
            for i in range(len(all_boundaries) - 1)
        ]

        for start, end in intervals:
            is_speech = False
            for vad_start, vad_end in vad_segments:
                if vad_start <= start and end <= vad_end:
                    is_speech = True
                    break
                if max(start, vad_start) < min(end, vad_end):
                    is_speech = True
                    break

            if not is_speech:
                continue

            best_speaker = None
            best_overlap = 0.0

            for speaker, segments in speaker_segments.items():
                for sp_start, sp_end in segments:
                    overlap_start = max(start, sp_start)
                    overlap_end = min(end, sp_end)
                    overlap = max(0.0, overlap_end - overlap_start)

                    if overlap > best_overlap:
                        best_overlap = overlap
                        best_speaker = speaker

            if best_speaker is None:
                best_speaker = self._find_nearest_speaker(start, speaker_segments)

            merged.append({"start": start, "end": end, "speaker_id": best_speaker})

        return self._merge_adjacent_segments(merged)

    def _find_nearest_speaker(
        self, time: float, speaker_segments: dict[str, list[tuple[float, float]]]
    ) -> str | None:
        """Находит говорящего, ближайшего по времени. Возвращает None, если спикеров нет вовсе."""
        best_speaker = None
        min_distance = float("inf")

        for speaker, segments in speaker_segments.items():
            for start, end in segments:
                if time < start:
                    distance = start - time
                elif time > end:
                    distance = time - end
                else:
                    distance = 0

                if distance < min_distance:
                    min_distance = distance
                    best_speaker = speaker

        return best_speaker

    def _merge_adjacent_segments(self, segments: list[dict]) -> list[dict]:
        """Объединяет соседние сегменты с одинаковым speaker_id"""
        if not segments:
            return []

        merged = []
        current = segments[0].copy()

        for seg in segments[1:]:
            if (
                seg["speaker_id"] == current["speaker_id"]
                and seg["start"] <= current["end"] + self.MERGE_ADJACENT_GAP_SEC
            ):
                current["end"] = max(current["end"], seg["end"])
            else:
                merged.append(current)
                current = seg.copy()

        merged.append(current)
        return merged


    def _build_json(
        self,
        audio_path: str,
        segments: list[dict],
        speaker_embeddings: dict[str, list[float] | None],
    ) -> dict:
        """Собирает итоговый json-файл"""
        result = {"audio_path": audio_path, "segments": []}

        for idx, seg in enumerate(segments):
            start = seg["start"]
            end = seg["end"]
            text = seg.get("text", "")
            speaker_id = seg.get("speaker_id")
            speaker_embedding = speaker_embeddings.get(speaker_id) if speaker_id is not None else None

            parts_info = [
                {
                    "start_time_ms": (word_info["start"] + start) * 1000,
                    "end_time_ms": (word_info["end"] + start) * 1000,
                    "text": word_info["word"],
                }
                for word_info in seg.get("words", [])
            ]

            result["segments"].append(
                {
                    "duration": end - start,
                    "model_name": self.model_name,
                    "original_parts_info": parts_info,
                    "speaker_embedding": speaker_embedding,
                    "speaker_id": speaker_id,
                    "start_time": start,
                    "end_time": end,
                    "text": text,
                    "uid": str(idx),
                }
            )

        return result


    def _process(self, audio_path: str) -> dict:
        """Запускает процесс соединения VAD, диаризации и ASR"""

        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)

        vad_segments = self.vad.process(audio)
        speaker_segments = self.diarizer.process(audio, sr)

        speaker_embeddings = self._create_speaker_embedding(audio, speaker_segments, sr)
        merged_segments = self._merge_segments_quality(vad_segments, speaker_segments)

        for idx, seg in enumerate(merged_segments):
            start = seg["start"]
            end = seg["end"]
            duration = end - start

            if duration <= self.MIN_ASR_SEGMENT_SEC:
                continue
            
            start_sample = int(start * sr)
            end_sample = int(end * sr)
            segment_audio = audio[start_sample:end_sample]

            temp_path = self.temp_dir / f"temp_segment_{idx}_{start:.3f}_{end:.3f}.wav"

            try:
                sf.write(str(temp_path), segment_audio, sr)
                asr_result = self.asr.transcribe_audio_gigaam(str(temp_path))
                seg["text"] = asr_result["text"]
                seg["words"] = asr_result.get("words", [])
            except Exception:
                logger.exception(
                    "Не удалось распознать сегмент %s [%.2f-%.2f] в %s",
                    idx, start, end, audio_path,
                )
                seg["text"] = ""
                seg["words"] = []
            finally:
                if idx % self.GPU_CLEAR_EVERY_N_SEGMENTS == 0:
                    self._clear_gpu()
                if temp_path.exists():
                    temp_path.unlink()

        return self._build_json(audio_path, merged_segments, speaker_embeddings)

    def _clear_gpu(self):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    def process_and_save(self, audio_path: str, out_folder: str = "results") -> Path:
        """Обрабатывает файл и сохраняет результат в json"""
        result = self._process(audio_path)

        out_dir = Path(out_folder)
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / f"{Path(audio_path).stem}_result.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        logger.info("Сохранено: %s", out_path)
        return out_path


In [7]:
def main():
    MODEL_NAME = "v3_e2e_rnnt"

    pipeline = AudioProcessingPipeline(HF_TOKEN, MODEL_NAME)
    logger.info("Устройство: %s", pipeline.device)

    current_dir = Path.cwd()
    path2data = current_dir / "data"

    if not path2data.exists():
        raise FileNotFoundError(f"Папка с данными не найдена: {path2data}")

    wav_files = sorted(
        f for f in os.listdir(path2data) if f.lower().endswith(".wav")
    )

    if not wav_files:
        logger.warning("В %s не найдено .wav файлов", path2data)
        return

    failed = []
    for audio_file in tqdm(wav_files, desc="Обработка аудио"):
        full_path = path2data / audio_file
        try:
            pipeline.process_and_save(str(full_path))
        except Exception:
            logger.exception("Ошибка при обработке %s", full_path)
            failed.append(audio_file)

    if failed:
        logger.warning("Не удалось обработать %d файл(ов): %s", len(failed), failed)
    else:
        logger.info("Готово: обработано %d файл(ов)", len(wav_files))


In [8]:
if __name__ == "__main__":
    main()


2026-08-12 02:18:26,376 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/config.yaml "HTTP/1.1 200 OK"
2026-08-12 02:18:27,939 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/segmentation/pytorch_model.bin "HTTP/1.1 302 Found"
2026-08-12 02:18:28,522 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/plda/xvec_transform.npz "HTTP/1.1 302 Found"
2026-08-12 02:18:28,688 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/plda/plda.npz "HTTP/1.1 302 Found"
2026-08-12 02:18:28,996 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/embedding/pytorch_model.bin "HTTP/1.1 302 Found"
2026-08-12 02:18:33,690 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/pyannote/embedding/resolve/main/pytorch